# 07. Land Value Capture (LVC) Városfejlesztési Szimuláció

**Cél**: A tervezett Mázsa téri városrehabilitáció által a környező magáningatlanokban generált értéknövekmény számszerűsítése és a közösségi értékvisszanyerés (LVC) 20 éves DCF pénzügyi szimulációja.

---

### 📖 Mi az a Land Value Capture (LVC)?
A közösségi beruházások (pl. sportközpont, parkosítás) felértékelik a környező magánlakásokat. Az LVC célja, hogy ezen értéktöbblet egy méltányos részét a város visszanyerje és abból finanszírozza a közberuházást.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()

# Alapértelmezett beruházási szintek
TIERS = {
    'Tier 1: Gyalogos átjárók & megálló': {'capex': 2.5e9, 'evek': 2, 'premium_pct': 0.05, 'leiras': 'Közvetlen gyalogos kapcsolatok'},
    'Tier 2: Tier 1 + Városi Park & Zöld': {'capex': 5.0e9, 'evek': 3, 'premium_pct': 0.10, 'leiras': 'Környezeti zöldinfrastruktúra'},
    'Tier 3: Intermodális Csomópont + Sport': {'capex': 15.0e9, 'evek': 5, 'premium_pct': 0.20, 'leiras': 'Komplex városmegújítás'}
}
print("LVC modell inicializálva.")

LVC modell inicializálva.


### 1. Beruházási Szintek (CAPEX) és Érintett Vagyon

**📌 Mit látunk a szcenáriókban?**
Három beruházási szintet modellezünk:
- **Tier 1 (Közterület-rendezés)**: 1.5 milliárd Ft CAPEX $\rightarrow$ +5% lokális felértékelődés.
- **Tier 2 (Komplex városi park & zöldinfrastruktúra)**: 4.5 milliárd Ft CAPEX $\rightarrow$ +12% felértékelődés.
- **Tier 3 (Mázsa téri Sportközpont & Csomópont)**: 12.0 milliárd Ft CAPEX $\rightarrow$ +22% felértékelődés.

**🔍 Mekkora a hatásterület vagyontömege?**
A Mázsa tér 15 perces gyalogos vonzáskörzetében lévő lakásállomány összértéke meghaladja a 100 milliárd forintot, így még mérsékelt felértékelődés is milliárdos értéktöbbletet generál.

In [2]:
# Érintett ingatlanállomány becslése a Mázsa tér 15 perces izokrónjában (KSH 2022 bázison: ~42 150 kerületi lakás 28,5%-a)
erintett_lakasok_becsles = 12000  # 12 000 lakás a 15p gyalogos zónában
atlag_lakasar = elado['price_huf'].median()
erintett_vagyon = erintett_lakasok_becsles * atlag_lakasar

tier2 = TIERS['Tier 2: Tier 1 + Városi Park & Zöld']
generalt_erteknovekmeny = erintett_vagyon * tier2['premium_pct']
capture_rate = 0.20
visszanyert_bevetel = generalt_erteknovekmeny * capture_rate
netto_onkormanyzati_egyenleg = visszanyert_bevetel - tier2['capex']
roi_pct = (visszanyert_bevetel / tier2['capex']) * 100

kpi_cards = [
    ("Beruházási Költség (CAPEX)", fmt_mft(tier2['capex'] / 1e6), "Önkormányzati költség", "#ef4444"),
    ("Generált Értéknövekmény", fmt_mft(generalt_erteknovekmeny / 1e6), "Magánvagyon bővülés", "#10b981"),
    ("LVC Visszanyerés (20%)", fmt_mft(visszanyert_bevetel / 1e6), "Közösségi bevétel", "#2563eb"),
    ("Önkormányzati Megtérülés", f"{roi_pct:.1f}%", "LVC / CAPEX arány", "#059669"),
    ("Nettó Közösségi Egyenleg", fmt_mft(netto_onkormanyzati_egyenleg / 1e6), "CAPEX levonása után", "#7c3aed"),
    ("Érintett Lakásállomány", f"{erintett_lakasok_becsles:,} db".replace(',', ' '), "15p sétaövezet", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kumulált Pénzáram és Megtérülési Idő (Cash Flow Profil)

**📌 Hogyan alakul a beruházás pénzügyi egyenlege?**
A 20 éves modellben a kezdeti kivitelezési évek negatív cash flow-ját a 4. évtől beérkező értéknövekményi hozzájárulások fordítják pozitívba.
- A projekt 20%-os visszanyerési kulcs mellett a 8-10. év környékén éri el a megtérülési pontot (Break-even).

In [3]:
evek = np.arange(0, 21)
cf_alap = np.zeros(21)
cf_alap[1:4] = -tier2['capex'] / 3.0  # 3 éves beruházás
# Éves LVC bevételek az 4. évtől fokozatosan 15 éven át
cf_alap[4:19] = visszanyert_bevetel / 15.0

cum_cf = np.cumsum(cf_alap)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=evek, y=cum_cf / 1e6,
    mode='lines+markers',
    name='Kumulált Cash Flow (Alap szcenárió)',
    line=dict(color='#2563eb', width=3),
    marker=dict(size=8)
))
fig1.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Megtérülési Küszöb (Break-even)')
fig1.update_layout(
    title='LVC Beruházás Kumulált Pénzárama (Millió Ft)',
    xaxis_title='Évek',
    yaxis_title='Kumulált Egyenleg (M Ft)',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig1.show()

### 3. Érzékenységvizsgálati Mátrix (Diszkontráta vs. Visszanyerési Kulcs)

**📌 Mit mutat a hőtérkép?**
Az NPV (Nettó Jelenérték) alakulását különböző kamatkörnyezetben (3-8% diszkontráta) és visszanyerési kulcsok (10-35%) mellett.
- Zöld mezők: gazdaságilag megtérülő, nyereséges kimenetelek.
- Piros mezők: veszteséges szcenáriók alacsony visszanyerés mellett.

In [4]:
rates = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08]
captures = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

npv_matrix = np.zeros((len(rates), len(captures)))
for i, r in enumerate(rates):
    for j, c in enumerate(captures):
        cf = np.zeros(21)
        cf[1:4] = -tier2['capex'] / 3.0
        cf[4:19] = (generalt_erteknovekmeny * c) / 15.0
        npv = sum(cf[t] / ((1 + r) ** t) for t in range(len(cf)))
        npv_matrix[i, j] = npv / 1e6

fig2 = px.imshow(
    npv_matrix,
    x=[f'{int(c*100)}%' for c in captures],
    y=[f'{int(r*100)}%' for r in rates],
    labels=dict(x="LVC Capture Rate (%)", y="Diszkontráta (%)", color="NPV (M Ft)"),
    text_auto='.0f',
    color_continuous_scale='RdYlGn',
    title='LVC Projekt Nettó Jelenértéke (NPV, M Ft) Érzékenységi Mátrixban',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 4. Interaktív LVC Döntéstámogató Szimulátor

**📌 Tesztelje a szcenáriókat!**
Válassza ki a beruházási szintet és állítsa be a paramétereket, hogy megnézze a projekt 20 éves nettó jelenértékét!

In [5]:
# Valós idejű Mázsa Tér Városfejlesztési Értéknövekmény (LVC) Szimulátor
html_lvc = f'''
<div id="lvc_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <div style="display:flex; justify-content:space-between; align-items:center; border-bottom:2px solid #f1f5f9; padding-bottom:12px; margin-bottom:18px;">
    <div>
      <h3 style="margin:0; color:#1e3a8a; font-size:19px; font-weight:700;">🏙️ Mázsa Tér Városfejlesztési Értéknövekmény (LVC) Szimulátor</h3>
      <p style="margin:3px 0 0 0; color:#64748b; font-size:13px;">Dinamikus Land Value Capture finanszírozási modellezés és közösségi megtérülés</p>
    </div>
    <span style="background:#dbeafe; color:#1d4ed8; font-size:11px; font-weight:700; padding:4px 10px; border-radius:9999px;">Kliensoldali JS Motor</span>
  </div>

  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(280px, 1fr)); gap:20px; margin-bottom:20px;">
    <!-- 1. oszlop: Fejlesztési Csomag -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">1. Beruházási Csomag (CAPEX)</div>

      <div style="margin-bottom:12px;">
        <label style="display:block; font-size:13px; font-weight:600; color:#475569; margin-bottom:6px;">Infrastruktúra Szint:</label>
        <select id="lvc_tier_sel" style="width:100%; padding:8px 10px; border-radius:6px; border:1px solid #cbd5e1; font-size:13px; background:#fff;" onchange="recalcLVC()">
          <option value="Tier 1: Csak Vasútállomás & Intermodális">Tier 1: Vasútállomás & Csomópont (15 Mrd Ft, +8% prémium)</option>
          <option value="Tier 2: Tier 1 + Városi Park & Zöld" selected>Tier 2: Vasútállomás + Park & Zöldfelület (25 Mrd Ft, +14% prémium)</option>
          <option value="Tier 3: Teljes TOD Akcióterület & Városközpont">Tier 3: Teljes TOD Városközpont (40 Mrd Ft, +22% prémium)</option>
        </select>
      </div>

      <div style="background:#eff6ff; border-left:4px solid #2563eb; padding:10px 12px; border-radius:4px; font-size:12px; color:#1e40af; margin-top:14px;">
        📍 <b>Érintett Ingatlanállomány:</b> {erintett_vagyon / 1e9:.1f} Mrd Ft magánvagyon a Mázsa tér 15 perces gyalogos elérhetőségi zónájában.
      </div>
    </div>

    <!-- 2. oszlop: LVC Paraméterek -->
    <div style="background:#f8fafc; padding:16px; border-radius:8px; border:1px solid #e2e8f0;">
      <div style="font-weight:700; color:#334155; margin-bottom:12px; font-size:14px; text-transform:uppercase; letter-spacing:0.5px;">2. Finanszírozási és Elvonási Kulcsok</div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Értéknövekmény elvonási kulcs (Capture Rate):</span> <span id="lvc_lbl_cap" style="color:#2563eb; font-weight:700;">20%</span>
        </div>
        <input type="range" id="lvc_cap" min="5" max="40" value="20" step="5" style="width:100%; accent-color:#2563eb;" oninput="recalcLVC()">
      </div>

      <div style="margin-bottom:14px;">
        <div style="display:flex; justify-content:space-between; font-size:13px; font-weight:600; color:#475569;">
          <span>Éves társadalmi diszkontráta:</span> <span id="lvc_lbl_disc" style="color:#7c3aed; font-weight:700;">5.0%</span>
        </div>
        <input type="range" id="lvc_disc" min="20" max="100" value="50" step="5" style="width:100%; accent-color:#7c3aed;" oninput="recalcLVC()">
      </div>

      <div style="font-size:11px; color:#64748b;">
        * Az LVC (Land Value Capture) mechanizmus célja, hogy a közpénzből megvalósuló infrastruktúra által generált magánvagyoni externália egy részét visszajuttassa a beruházás finanszírozására.
      </div>
    </div>
  </div>

  <!-- KPI Kártyák -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:10px;">
    <div style="background:#f0fdf4; border:1px solid #86efac; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#166534; text-transform:uppercase;">Szimulált Projekt NPV</div>
      <div id="lvc_res_npv" style="font-size:26px; font-weight:800; color:#15803d; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:11px; color:#16a34a;">Önkormányzati diszkontált mérleg</div>
    </div>

    <div style="background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Visszanyert Közösségi Forrás</div>
      <div id="lvc_res_rec" style="font-size:26px; font-weight:800; color:#1d4ed8; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:11px; color:#2563eb;">15 éves kumulált bevétel</div>
    </div>

    <div style="background:#faf5ff; border:1px solid #d8b4fe; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#6b21a8; text-transform:uppercase;">CAPEX Fedezeti Arány</div>
      <div id="lvc_res_cov" style="font-size:24px; font-weight:800; color:#7e22ce; margin:4px 0;">--%</div>
      <div style="font-size:11px; color:#9333ea;">Beruházás megtérülési hányad</div>
    </div>

    <div style="background:#fffbeb; border:1px solid #fde68a; border-radius:10px; padding:14px 18px; text-align:center;">
      <div style="font-size:11px; font-weight:700; color:#92400e; text-transform:uppercase;">Magánvagyoni Externália</div>
      <div id="lvc_res_gain" style="font-size:24px; font-weight:800; color:#b45309; margin:4px 0;">-- Mrd Ft</div>
      <div style="font-size:11px; color:#d97706;">Ingatlanfelértékelődés összege</div>
    </div>
  </div>
</div>

<script>
(function() {{
  const erintett = {float(erintett_vagyon)};
  const tiers = {{
    "Tier 1: Csak Vasútállomás & Intermodális": {{ capex: 15e9, premium: 0.08, evek: 3 }},
    "Tier 2: Tier 1 + Városi Park & Zöld": {{ capex: 25e9, premium: 0.14, evek: 4 }},
    "Tier 3: Teljes TOD Akcióterület & Városközpont": {{ capex: 40e9, premium: 0.22, evek: 5 }}
  }};

  function update() {{
    const tierKey = document.getElementById('lvc_tier_sel').value;
    const t_info = tiers[tierKey] || tiers["Tier 2: Tier 1 + Városi Park & Zöld"];
    const cap_rate = parseFloat(document.getElementById('lvc_cap').value) / 100;
    const disc = parseFloat(document.getElementById('lvc_disc').value) / 1000;

    document.getElementById('lvc_lbl_cap').innerText = Math.round(cap_rate * 100) + '%';
    document.getElementById('lvc_lbl_disc').innerText = (disc * 100).toFixed(1) + '%';

    const gain = erintett * t_info.premium;
    const rec = gain * cap_rate;

    let npv = 0;
    for (let t = 1; t <= t_info.evek; t++) {{
      npv -= (t_info.capex / t_info.evek) / Math.pow(1.0 + disc, t);
    }}
    for (let t = t_info.evek + 1; t <= t_info.evek + 15; t++) {{
      npv += (rec / 15.0) / Math.pow(1.0 + disc, t);
    }}

    const capex_cov = (rec / t_info.capex) * 100;

    document.getElementById('lvc_res_npv').innerText = (npv > 0 ? '+' : '') + (npv / 1e9).toFixed(1) + ' Mrd Ft';
    document.getElementById('lvc_res_rec').innerText = (rec / 1e9).toFixed(1) + ' Mrd Ft';
    document.getElementById('lvc_res_cov').innerText = capex_cov.toFixed(1) + '%';
    document.getElementById('lvc_res_gain').innerText = (gain / 1e9).toFixed(1) + ' Mrd Ft';
  }}

  window.recalcLVC = update;
  setTimeout(update, 50);
}})();
</script>
'''
display(HTML(html_lvc))